In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/Research"

/content/drive/MyDrive/Colab Notebooks/Research


In [ ]:
# ============================================================
# USOM COLLECTION — WITH STREAMING + RETRY
# ============================================================
!pip install tldextract
import re, warnings, time, requests, tldextract
import numpy as np, pandas as pd
from datetime import datetime
from collections import Counter
from io import StringIO

warnings.filterwarnings("ignore")
import urllib3; urllib3.disable_warnings()

# ── CONFIG ───────────────────────────────────────────────────
USOM_URL    = "https://www.usom.gov.tr/url-list.txt"
OUTPUT_CSV  = "new+with_new_keywords_usom_collected.csv"
MAX_COLLECT = 471286


TURKISH_KEYWORDS = [
    # Authentication / Account
    "giris", "giriş", "login", "signin", "oturum",
    "hesap", "hesabim", "hesabım", "account",
    "uye", "üye", "uyelik", "üyelik", "abonelik",

    # Payment / Banking
    "odeme", "ödeme", "payment", "banka", "bank",
    "iban", "swift", "havale", "eft", "transfer",
    "para", "para-transferi", "kredi", "kart",
    "kredi-karti", "kredi karti", "debit", "visa",
    "mastercard", "pos", "islem", "işlem",

    # Security / Verification
    "dogrula", "doğrula", "verification", "verify",
    "guvenli", "güvenli", "security", "secure",
    "sifre", "şifre", "parola", "password",
    "sms", "otp", "kod", "onay", "onayla",
    "iki-asamali", "2fa", "kimlik-dogrulama",

    # Identity / Government
    "tckimlik", "tc-kimlik", "kimlik",
    "pasaport", "ehliyet", "edevlet", "e-devlet",
    "sgk", "vergidairesi", "nvi", "mhrs",

    # Shopping / Logistics
    "kargo", "teslimat", "takip", "siparis",
    "sipariş", "fatura", "irsaliye",
    "kampanya", "indirim", "kupon", "promosyon",
    "hediye", "cekilis", "çekiliş",
    "odul", "ödül", "kazandin", "kazandın",
    "stok", "satis", "satış", "urun", "ürün",

    # Crypto / Finance
    "kripto", "crypto", "bitcoin", "btc",
    "ethereum", "eth", "binance", "wallet",
    "cuzdan", "cüzdan",

    # Healthcare
    "saglik", "sağlık", "hastane", "randevu",
    "ilac", "ilaç", "eczane",

    # Misc
    "destek", "musteri", "müşteri",
    "yardim", "yardım", "online", "mobil",
]

TURKISH_TARGETS = [
    # Banks
    "garanti", "garantibbva", "akbank", "isbank",
    "işbank", "ziraat", "ziraatbank",
    "halkbank", "vakifbank", "vakıfbank",
    "yapikredi", "yapıkredi", "denizbank",
    "finansbank", "qnb", "qnbfinansbank",
    "teb", "enpara", "kuveytturk",
    "albaraka", "vakifkatilim", "ziraatkatilim",
    "ing-bank", "ingbank", "ing.com.tr",

    # E-commerce
    "trendyol", "hepsiburada", "gittigidiyor",
    "n11", "amazon.com.tr", "amazon-tr",
    "ciceksepeti", "çiçeksepeti",
    "morhipo", "boyner", "lcwaikiki",
    "lcw", "lc-waikiki", "defacto",
    "koton", "beymen", "mavi",
    "colins", "flo", "deichmann",

    # Government
    "edevlet", "e-devlet", "turkiye",
    "türkiye", "gov.tr", "mhrs",
    "sgk", "geliridaresi", "uyap",

    # Cargo / Shipping
    "ptt", "ptt-kargo", "dhl",
    "aras", "aras-kargo", "yurtici",
    "yurtiçi", "mng", "ups", "fedex",
    "suratkargo", "süratkargo",

    # Telecom
    "turkcell", "vodafone",
    "turk-telekom", "turktelekom",
    "bimcell", "turksat",

    # Retail / Grocery
    "bim", "bim.com", "bim-",
    "-bim", "migros", "carrefour",
    "metro", "a101", "sok",
    "şok", "hakmar",

    # Airlines / Travel
    "thy", "turkishairlines",
    "pegasus", "sunexpress",
    "obilet", "enuygun",

    # Crypto / Exchanges
    "btcturk", "paribu", "bitexen",
    "binance-tr", "okx", "bybit",

    # Tech / Payment
    "papara", "ininal", "paycell",
    "iyzico", "shopier",

    # Misc popular brands
    "ayakkabi", "ayakkabı",
    "getir", "yemeksepeti",
    "hepsipay", "trgoals",
]


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/plain,text/html,*/*",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
}

print("=" * 60)
print("USOM COLLECTION — STREAMING + RETRY")
print(f"  Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 60)

# ── 1. DOWNLOAD WITH STREAMING ───────────────────────────────
print("\n[1] Downloading USOM list (streaming) ...")

def download_usom(url: str, max_retries: int = 5) -> list:
    """
    Stream download with retry — handles large files and
    USOM server timeouts gracefully.
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"    Attempt {attempt}/{max_retries} ...")
            session = requests.Session()
            session.headers.update(HEADERS)

            # stream=True — reads in chunks, avoids single timeout
            resp = session.get(
                url,
                stream=True,
                timeout=(15, 120),   # (connect_timeout, read_timeout)
                verify=False,
            )
            resp.raise_for_status()

            chunks = []
            total_bytes = 0
            for chunk in resp.iter_content(chunk_size=65536):
                if chunk:
                    chunks.append(chunk)
                    total_bytes += len(chunk)
                    if total_bytes % (1024 * 1024) < 65536:
                        print(f"      {total_bytes / 1024 / 1024:.1f} MB downloaded ...",
                              end="\r")

            raw_text = b"".join(chunks).decode("utf-8", errors="replace")
            lines = raw_text.strip().split("\n")
            print(f"\n    Downloaded: {len(lines):,} lines "
                  f"({total_bytes/1024/1024:.1f} MB)")
            return lines

        except Exception as e:
            print(f"    Attempt {attempt} failed: {type(e).__name__}: {e}")
            if attempt < max_retries:
                wait = 10 * attempt
                print(f"    Retrying in {wait}s ...")
                time.sleep(wait)
            else:
                print("    All attempts failed.")
                return []

raw_lines = download_usom(USOM_URL)

# ── FALLBACK: try alternative USOM endpoints ─────────────────
if not raw_lines:
    print("\n    Trying alternative USOM endpoints ...")
    alternatives = [
        "http://www.usom.gov.tr/url-list.txt",    # HTTP instead of HTTPS
        "https://usom.gov.tr/url-list.txt",        # no www
    ]
    for alt_url in alternatives:
        print(f"    Trying: {alt_url}")
        raw_lines = download_usom(alt_url, max_retries=2)
        if raw_lines:
            print(f"    Success with: {alt_url}")
            break


# ── 2. CLEAN ─────────────────────────────────────────────────
print("\n[2] Cleaning ...")

def clean_url(raw: str):
    raw = raw.strip().lstrip("#").strip()
    if not raw or raw.startswith("#") or len(raw) < 5:
        return None
    raw = re.sub(r'^https?://', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^www\.', '',  raw, flags=re.IGNORECASE)
    if " " in raw or "\t" in raw:
        return None
    return raw.lower().strip("/")

cleaned = []
for line in raw_lines:
    u = clean_url(line)
    if u:
        cleaned.append(u)

cleaned = list(dict.fromkeys(cleaned))
print(f"    After cleaning: {len(cleaned):,} unique URLs")

# ── 3. SCORE ─────────────────────────────────────────────────
print("\n[3] Scoring for Turkish targeting ...")

def score_url(url: str) -> dict:
    u = url.lower()
    n_kw  = sum(k in u for k in TURKISH_KEYWORDS)
    n_tgt = sum(t in u for t in TURKISH_TARGETS)
    has_tr = int(
        u.endswith(".tr") or ".tr/" in u or
        ".com.tr" in u or ".net.tr" in u or
        ".org.tr" in u or ".gov.tr" in u or
        ".edu.tr" in u or ".bel.tr" in u
    )
    score = has_tr * 3 + n_kw * 2 + n_tgt * 2
    return {
        "url":            url,
        "label":          "malicious",
        "source":         "USOM",
        "tr_score":       score,
        "has_tr_tld":     has_tr,
        "has_turkish_kw": int(n_kw > 0),
        "n_turkish_kw":   n_kw,
        "has_turkish_tgt":int(n_tgt > 0),
        "n_target_signals": n_tgt,
        "is_reachable":   -1,
        "http_status":    -1,
    }

records  = [score_url(u) for u in cleaned]
score_df = pd.DataFrame(records).sort_values(
    "tr_score", ascending=False
).reset_index(drop=True)

tr_strong = score_df[score_df["tr_score"] >= 4]
tr_medium = score_df[(score_df["tr_score"] >= 2) &
                      (score_df["tr_score"] < 4)]
tr_weak   = score_df[score_df["tr_score"] < 2]

print(f"    High-confidence Turkish (≥4) : {len(tr_strong):,}")
print(f"    Medium-confidence Turkish (≥2): {len(tr_medium):,}")
print(f"    Non-Turkish / generic        : {len(tr_weak):,}")

print(f"\n    Top 15 Turkish keywords found:")
kw_counts = Counter()
for u in score_df["url"]:
    for kw in TURKISH_KEYWORDS:
        if kw in u:
            kw_counts[kw] += 1
for kw, cnt in kw_counts.most_common(15):
    print(f"      {kw:<22s}: {cnt:>6,}")

print(f"\n    Top 10 brand/institution targets:")
tgt_counts = Counter()
for u in score_df["url"]:
    for t in TURKISH_TARGETS:
        if t in u:
            tgt_counts[t] += 1
for tgt, cnt in tgt_counts.most_common(10):
    print(f"      {tgt:<22s}: {cnt:>6,}")

# ── 4. SELECT ────────────────────────────────────────────────
print(f"\n[4] Selecting up to {MAX_COLLECT:,} URLs ...")

priority = pd.concat(
    [tr_strong, tr_medium, tr_weak], ignore_index=True
).head(MAX_COLLECT)

n_s = min(len(tr_strong), MAX_COLLECT)
n_m = min(len(tr_medium), max(0, MAX_COLLECT - len(tr_strong)))
n_w = max(0, MAX_COLLECT - len(tr_strong) - len(tr_medium))
print(f"    Total selected           : {len(priority):,}")
print(f"      High-confidence (≥4)   : {n_s:,}")
print(f"      Medium-confidence (≥2) : {n_m:,}")
print(f"      Generic (<2)           : {n_w:,}")

# ── 5. SAVE ──────────────────────────────────────────────────
print(f"\n[5] Saving ...")
priority.to_csv(OUTPUT_CSV, index=False)
print(f"    Saved: {OUTPUT_CSV}  ({len(priority):,} rows)")

# ── 6. SUMMARY ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("COLLECTION COMPLETE")
print("=" * 60)
print(f"""
  Downloaded          : {len(raw_lines):,} entries
  After deduplication : {len(cleaned):,} unique URLs
  High-confidence (≥4): {len(tr_strong):,}
  Medium (≥2)         : {len(tr_medium):,}
  Saved               : {len(priority):,}

  Top keywords:""")
for kw, cnt in kw_counts.most_common(10):
    print(f"    {kw:<20s} {cnt:>6,}  {'█' * (cnt // 500)}")

print(f"""
  NEXT STEPS:
  1. Upload {OUTPUT_CSV} to Colab
  2. Run Step 1B integration script
  3. Rerun Steps 1-8 with dataset_with_folds_usom.csv
""")
print("=" * 60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.0 MB/s eta 0:00:00
USOM COLLECTION — STREAMING + RETRY
  Date: 2026-05-14 16:45

[1] Downloading USOM list (streaming) ...
    Attempt 1/5 ...
    Attempt 1 failed: ChunkedEncodingError: Response ended prematurely
    Retrying in 10s ...
    Attempt 2/5 ...
    Attempt 2 failed: ChunkedEncodingError: Response ended prematurely
    Retrying in 20s ...
    Attempt 3/5 ...

    Downloaded: 472,237 lines (10.1 MB)

[2] Cleaning ...
    After cleaning: 471,944 unique URLs

[3] Scoring for Turkish targeting ...
    High-confidence Turkish (≥4) : 63,160
    Medium-confidence Turkish (≥2): 97,986
    Non-Turkish / generic        : 310,798

    Top 15 Turkish keywords found:
      online                : 33,923
      bank                  : 14,410
      giris                 : 11,163
      kampanya              : 10,820
      mobil                 : 10,088
      kredi                 :  7,590
      odeme                 :  7,107
     

In [ ]:
# ============================================================
# STEP 1B — INTEGRATE USOM INTO MAIN DATASET (run in Colab)
# ============================================================
# Upload usom_collected.csv to Colab first, then run this.
# ============================================================

import pandas as pd
import numpy as np
import tldextract
import pickle

print("=" * 60)
print("STEP 1B — USOM INTEGRATION")
print("=" * 60)

# ── Load both datasets ────────────────────────────────────────
existing = pd.read_csv("dataset_with_folds.csv")
usom_raw = pd.read_csv("new_usom_collected.csv")

print(f"\n[1] Existing dataset : {len(existing):,} rows")
print(f"    USOM collected   : {len(usom_raw):,} rows")

# Existing class counts
e_ben = (existing["label"] == "benign").sum()
e_mal = (existing["label"] == "malicious").sum()
print(f"\n    Existing — benign: {e_ben:,}  malicious: {e_mal:,}")

# ── Deduplicate USOM against existing ────────────────────────
existing_domains = set(existing["domain"].str.lower().str.strip("/"))
usom_raw["domain"] = usom_raw["url"].str.lower().str.strip("/")
usom_new = usom_raw[~usom_raw["domain"].isin(existing_domains)].copy()
print(f"    USOM new (not in existing): {len(usom_new):,}")

# Turkish keyword breakdown of new URLs
tr_kw   = usom_new["has_turkish_kw"].sum()
tr_tgt  = usom_new["has_turkish_tgt"].sum()
live    = usom_new["is_reachable"].sum()
print(f"\n    Turkish keyword URLs : {int(tr_kw):,}")
print(f"    Turkish target URLs  : {int(tr_tgt):,}")
print(f"    Live / reachable     : {int(live):,}")

# ── Strategy: add ALL high-confidence Turkish, sample the rest
tr_strong_new = usom_new[usom_new["tr_score"] >= 4]
tr_other_new  = usom_new[usom_new["tr_score"] < 4]

print(f"\n[2] Integration strategy:")
print(f"    Include ALL high-confidence Turkish: {len(tr_strong_new):,}")
print(f"    Sample from remaining USOM         : up to "
      f"{max(0, e_mal - len(tr_strong_new)):,}")

# Fill up to match current malicious count (keep 50/50)
# Or optionally go larger — adjust n_target as needed
n_target    = max(e_ben, e_mal + len(tr_strong_new))
n_from_other = max(0, n_target - e_mal - len(tr_strong_new))

other_sample = tr_other_new.sample(
    n=min(n_from_other, len(tr_other_new)),
    random_state=42
) if n_from_other > 0 else pd.DataFrame()

usom_to_add = pd.concat([tr_strong_new, other_sample],
                         ignore_index=True)

print(f"    Total USOM URLs to add: {len(usom_to_add):,}")

# ── Build combined dataset ────────────────────────────────────
usom_add_clean = pd.DataFrame({
    "domain":    usom_to_add["domain"],
    "label":     "malicious",
    "label_enc": 1,
    "source":    "USOM",
})

existing["source"] = "original"
combined = pd.concat(
    [existing[["domain","label","label_enc","source"]],
     usom_add_clean],
    ignore_index=True
).drop_duplicates(subset=["domain"])

# Rebalance: undersample to 50/50 if needed
n_ben = (combined["label"] == "benign").sum()
n_mal = (combined["label"] == "malicious").sum()
n_min = min(n_ben, n_mal)

ben_df = combined[combined["label"] == "benign"].sample(
    n=n_min, random_state=42)
mal_df = combined[combined["label"] == "malicious"]

# Within malicious — prioritise Turkish USOM
usom_mal = mal_df[mal_df["source"] == "USOM"]
orig_mal  = mal_df[mal_df["source"] == "original"]

if len(usom_mal) >= n_min:
    mal_sample = usom_mal.sample(n=n_min, random_state=42)
else:
    n_orig = n_min - len(usom_mal)
    mal_sample = pd.concat([
        usom_mal,
        orig_mal.sample(n=min(n_orig, len(orig_mal)), random_state=42)
    ])

balanced = pd.concat([ben_df, mal_sample]).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print(f"\n[3] Balanced dataset:")
print(f"    Total rows    : {len(balanced):,}")
print(balanced["label"].value_counts().to_string())
usom_pct = (balanced[balanced["label"]=="malicious"]["source"]
            == "USOM").mean() * 100
print(f"    USOM share of malicious: {usom_pct:.1f}%")

# ── Domain-aware fold assignment ─────────────────────────────
print("\n[4] Assigning domain-aware folds ...")

def get_reg_domain(url):
    try:
        u = str(url).strip()
        if not u.startswith(("http://","https://")):
            u = "http://" + u
        ext = tldextract.extract(u)
        if ext.domain and ext.suffix:
            return f"{ext.domain}.{ext.suffix}".lower()
        return u.lower()
    except Exception:
        return str(url).lower()

balanced["reg_domain"] = balanced["domain"].apply(get_reg_domain)

domains = balanced["reg_domain"].unique()
np.random.seed(42)
np.random.shuffle(domains)
domain_fold = {d: i % 5 for i, d in enumerate(domains)}
balanced["fold"] = balanced["reg_domain"].map(domain_fold)

print(f"    Fold distribution:")
print(balanced["fold"].value_counts().sort_index().to_string())

# Verify zero domain overlap
fold_domains = [
    set(balanced[balanced["fold"]==f]["reg_domain"])
    for f in range(5)
]
overlaps = []
for i in range(5):
    for j in range(i+1, 5):
        overlap = fold_domains[i] & fold_domains[j]
        if overlap:
            overlaps.append((i, j, len(overlap)))

if overlaps:
    print(f"    ⚠ Domain overlaps found: {overlaps}")
else:
    print(f"    Domain overlap across folds: 0 ✓")

# ── Save ─────────────────────────────────────────────────────
balanced.to_csv("dataset_with_folds_usom.csv", index=False)
print(f"\n[5] Saved: dataset_with_folds_usom.csv")

# ── What changed ─────────────────────────────────────────────
print(f"\n" + "=" * 60)
print("INTEGRATION COMPLETE")
print("=" * 60)

orig_mal_n = len(existing[existing["label"]=="malicious"])
new_mal_n  = (balanced["label"]=="malicious").sum()
usom_added = (balanced[balanced["label"]=="malicious"]["source"]
              == "USOM").sum()

print(f"""
  Original malicious URLs  : {orig_mal_n:,}
  USOM URLs added          : {usom_added:,}
  Final malicious count    : {new_mal_n:,}
  Final benign count       : {(balanced["label"]=="benign").sum():,}
  Final total              : {len(balanced):,}

  Expected impact on next pipeline run:
  ─────────────────────────────────────
  Turkish keyword SHAP     : ~0.00003 → expect >0.01
  Malicious content fetch  : 0%       → expect 10–30%
  is_tr_domain leakage     : may increase (more .tr malicious)
  Overall AUC              : may drop slightly (harder problem)
  Thesis Turkish claim     : now empirically validated

  Next → rename dataset_with_folds_usom.csv to
          dataset_with_folds.csv and rerun all steps
          OR update EXISTING_CSV variable in each step.
""")
print("=" * 60)

STEP 1B — USOM INTEGRATION

[1] Existing dataset : 120,976 rows
    USOM collected   : 471,002 rows

    Existing — benign: 60,488  malicious: 60,488
    USOM new (not in existing): 411,603

    Turkish keyword URLs : 16,861
    Turkish target URLs  : 14,683
    Live / reachable     : -411,603

[2] Integration strategy:
    Include ALL high-confidence Turkish: 6,986
    Sample from remaining USOM         : up to 53,502
    Total USOM URLs to add: 6,986

[3] Balanced dataset:
    Total rows    : 120,976
label
benign       60488
malicious    60488
    USOM share of malicious: 11.5%

[4] Assigning domain-aware folds ...
    Fold distribution:
fold
0    23671
1    23560
2    25086
3    23806
4    24853
    Domain overlap across folds: 0 ✓

[5] Saved: dataset_with_folds_usom.csv

INTEGRATION COMPLETE

  Original malicious URLs  : 60,488
  USOM URLs added          : 6,986
  Final malicious count    : 60,488
  Final benign count       : 60,488
  Final total              : 120,976

  Expected 